# 🚀 SRΨ-Engine Ablation Study - Google Colab

## 快速开始（3 步）

**⚠️ 第一步：启用 GPU**

点击菜单：**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

**⚠️ 第二步：运行下面的代码**

依次点击每个 cell 左边的 ▶️ 按钮（或按 Shift+Enter）

**⚠️ 第三步：选择实验**

在 "Step 5" 中修改 `model_name` 选择要运行的实验

---

## 📊 可用实验

- `srpsi_real`: SRΨ Real-only (Exp2)
- `srpsi_no_r`: SRΨ w/o Rhythm operator (Exp3)
- `conv_baseline`: Pure Convolution (Exp4)
- `transformer_rel_pe`: Transformer with Relative Position (Exp5)

## ⏱️ 预计时间（GPU）

- 单个实验：30-45 分钟
- 4 个实验并行：30-45 分钟（同时完成！）

## 💡 Pro Tip

**打开多个浏览器标签页，每个运行不同的实验，40 分钟后全部完成！**

## Step 1: 检查 GPU

In [1]:
import torch
import os

print('='*60)
print('GPU 状态检查')
print('='*60)
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'GPU 内存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('\n✅ GPU 已就绪！可以开始训练。')
else:
    print('\n❌ 未检测到 GPU！')
    print('\n请启用 GPU：')
    print('  1. 点击 Runtime → Change runtime type')
    print('  2. Hardware accelerator 选择 T4 GPU')
    print('  3. 点击 Save')
    print('  4. 重新运行此 cell')

GPU 状态检查
PyTorch 版本: 2.10.0+cu128
CUDA 可用: True
GPU 型号: Tesla T4
GPU 内存: 15.6 GB

✅ GPU 已就绪！可以开始训练。


## Step 2: 克隆仓库（使用 HTTPS，无需认证）

In [2]:
# 克隆仓库
!git clone https://github.com/Mozilla2004/srpsi-engine-tiny.git

# 进入项目目录
%cd srpsi-engine-tiny

# 验证克隆成功
print('\n✅ 仓库克隆成功！')
!ls -la
print('\n目录内容如上所示 ✓')

Cloning into 'srpsi-engine-tiny'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 79 (delta 20), reused 73 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 2.08 MiB | 4.42 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/srpsi-engine-tiny

✅ 仓库克隆成功！
total 228
drwxr-xr-x 6 root root  4096 Mar 14 12:04 .
drwxr-xr-x 1 root root  4096 Mar 14 12:04 ..
-rw-r--r-- 1 root root  3131 Mar 14 12:04 ABLATION_EVALUATION_GUIDE.md
-rw-r--r-- 1 root root  5203 Mar 14 12:04 ABLATION_PLAN.md
-rw-r--r-- 1 root root  4134 Mar 14 12:04 ABLATION_README.md
-rw-r--r-- 1 root root  4361 Mar 14 12:04 analyze_checkpoint.py
-rw-r--r-- 1 root root 10417 Mar 14 12:04 colab_ablation_study.ipynb
-rw-r--r-- 1 root root 11375 Mar 14 12:04 colab_fixed.ipynb
drwxr-xr-x 2 root root  4096 Mar 14 12:04 config
-rw-r--r-- 1 root root  8945 Mar 14 12:04 DELIVERY_SUMMARY.md
-rw-r--r-- 1 root root  

In [6]:
import os
os.makedirs('data', exist_ok=True)
print('✅ data 目录已创建')

✅ data 目录已创建


## Step 3: 安装依赖

In [3]:
# 安装必需的包
!pip install -q pyyaml tqdm

import yaml
from tqdm import tqdm

print('\n✅ 依赖安装完成！')
print('  ✓ pyyaml')
print('  ✓ tqdm')
print('  ✓ torch')


✅ 依赖安装完成！
  ✓ pyyaml
  ✓ tqdm
  ✓ torch


## Step 4: 生成训练数据

In [8]:
import os

data_file = 'data/burgers_1d.npy'

if os.path.exists(data_file):
    size_mb = os.path.getsize(data_file) / (1024 * 1024)
    print(f'\n✅ 数据文件已存在！')
    print(f'  文件: {data_file}')
    print(f'  大小: {size_mb:.1f} MB')
    print('  ✓ 准备开始训练')
else:
    print('\n⏳ 正在生成训练数据...')
    print('  这可能需要 2-3 分钟，请耐心等待...')
    !python src/data_gen.py

    if os.path.exists(data_file):
        size_mb = os.path.getsize(data_file) / (1024 * 1024)
        print(f'\n✅ 数据生成完成！')
        print(f'  大小: {size_mb:.1f} MB')
    else:
        print('❌ 数据生成失败，请检查错误信息')


⏳ 正在生成训练数据...
  这可能需要 2-3 分钟，请耐心等待...
Generating burgers_1d dataset...
  Samples: 4000
  Steps: 48
  Grid: 128 x 48
  Calculated dx: 0.049087 (matching 2π domain)
Generated 100/4000 samples
Generated 200/4000 samples
Generated 300/4000 samples
Generated 400/4000 samples
Generated 500/4000 samples
Generated 600/4000 samples
Generated 700/4000 samples
Generated 800/4000 samples
Generated 900/4000 samples
Generated 1000/4000 samples
Generated 1100/4000 samples
Generated 1200/4000 samples
Generated 1300/4000 samples
Generated 1400/4000 samples
Generated 1500/4000 samples
Generated 1600/4000 samples
Generated 1700/4000 samples
Generated 1800/4000 samples
Generated 1900/4000 samples
Generated 2000/4000 samples
Generated 2100/4000 samples
Generated 2200/4000 samples
Generated 2300/4000 samples
Generated 2400/4000 samples
Generated 2500/4000 samples
Generated 2600/4000 samples
Generated 2700/4000 samples
Generated 2800/4000 samples
Generated 2900/4000 samples
Generated 3000/4000 samples
Gener

## Step 5: 运行实验

**👇 在这里修改 `model_name` 选择要运行的实验**

可用选项：
- `srpsi_real`: SRΨ Real-only (Exp2) - 实值状态，无虚部
- `srpsi_no_r`: SRΨ w/o Rhythm (Exp3) - 无节律算子
- `conv_baseline`: Pure Convolution (Exp4) - 纯卷积基线
- `transformer_rel_pe`: Transformer Rel-PE (Exp5) - 相对位置编码

**💡 提示：打开多个标签页可以并行运行多个实验！**

In [ ]:
# ============================================================
# ⚙️ 配置您的实验
# ============================================================

model_name = 'conv_baseline'  # 修改这里选择实验
num_epochs = 80            # 训练轮数

# 可选值:
# - 'srpsi_real': 实值状态 (Exp2)
# - 'srpsi_no_r': 无节律算子 (Exp3)
# - 'conv_baseline': 纯卷积基线 (Exp4)
# - 'transformer_rel_pe': 相对位置编码 (Exp5)

# ============================================================

print('\n' + '='*60)
print(f'开始实验: {model_name}')
print(f'训练轮数: {num_epochs}')
print(f'输出目录: outputs/ablation_{model_name}')
print(f'预计时间: ~30-45 分钟 (GPU)')
print(f'"="*60\n')

# 运行训练
!python src/train.py \
  --config config/burgers.yaml \
  --model {model_name} \
  --data data/burgers_1d.npy \
  --output outputs/ablation_{model_name} \
  --epochs {num_epochs}


开始实验: conv_baseline
训练轮数: 80
输出目录: outputs/ablation_conv_baseline
预计时间: ~30-45 分钟 (GPU)
"="*60

2026-03-14 12:19:25.768233: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773490765.789628    5441 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773490765.796523    5441 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773490765.814627    5441 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773490765.814655    5441 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more t

## Step 6: 检查结果

In [ ]:
from glob import glob
import os

checkpoint_dir = f'outputs/ablation_{model_name}/{model_name}/checkpoints'

if os.path.exists(checkpoint_dir):
    checkpoints = glob(f'{checkpoint_dir}/best*.pt')

    if checkpoints:
        best_model = checkpoints[0]
        size_mb = os.path.getsize(best_model) / (1024 * 1024)
        print(f'\n✅ 训练完成！')
        print(f'\n最佳模型:')
        print(f'  文件: {best_model}')
        print(f'  大小: {size_mb:.2f} MB')
        print(f'\n下载方法:')
        print(f'  1. 点击左侧 📁 文件图标')
        print(f'  2. 导航到: srpsi-engine-tiny/outputs/ablation_{model_name}/{model_name}/checkpoints/')
        print(f'  3. 右键点击 best*.pt → Download')
    else:
        print(f'\n⚠️ 未找到最佳 checkpoint')
        print(f'\n可能还在训练中...')
        !ls -lh {checkpoint_dir}/
else:
    print(f'\n❌ 目录不存在: {checkpoint_dir}')
    print(f'\n请检查训练是否正常开始')

## 📖 快速参考

### 并行运行多个实验（推荐）

要在 ~40 分钟内完成所有 4 个实验：

1. **打开 4 个浏览器标签页**，每个都打开这个 notebook

2. **在每个标签页的 Step 5 中设置不同的实验**：
   - 标签 1: `model_name = 'srpsi_real'`
   - 标签 2: `model_name = 'srpsi_no_r'`
   - 标签 3: `model_name = 'conv_baseline'`
   - 标签 4: `model_name = 'transformer_rel_pe'`

3. **在每个标签页运行 Step 5**

4. **40 分钟后全部完成！** 🎉

---

### 训练时间预估（GPU）

| 实验 | 每个Epoch | 总时间（80 epochs） |
|------|----------|-------------------|
| SRΨ Real | ~20秒 | ~30分钟 |
| SRΨ w/o R | ~25秒 | ~35分钟 |
| Conv Baseline | ~15秒 | ~25分钟 |
| Transformer | ~30秒 | ~45分钟 |

---

### 故障排除

**问题**: GPU available: False
- **解决**: Runtime → Change runtime type → GPU → Save → 重新运行 Step 1

**问题**: 训练很慢（CPU）
- **解决**: 确认启用了 GPU，检查 Step 1 的输出

**问题**: Out of memory
- **解决**: Runtime → Restart runtime，然后重新运行

**问题**: Colab 自动休眠
- **解决**: 运行下面的防休眠脚本

## 💡 防止 Colab 休眠（可选）

In [ ]:
# 运行此 cell 可以防止 Colab 在长时间训练时自动断开
# (在训练过程中，此脚本会每分钟点击一次连接按钮)

%%javascript
function ClickConnect(){
    console.log("Colab保持连接中...");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)

print('✅ 防休眠脚本已激活！')
print('   (训练过程中 Colab 将保持连接)')

---

## 🎉 准备就绪！

现在您已经准备好在 Google Colab GPU 上运行 SRΨ-Engine ablation study 了！

**步骤回顾**:
1. ✅ 启用 GPU（Runtime → Change runtime type → GPU）
2. ✅ 运行 Step 1 检查 GPU
3. ✅ 运行 Step 2 克隆仓库
4. ✅ 运行 Step 3 安装依赖
5. ✅ 运行 Step 4 生成数据
6. ✅ 在 Step 5 选择实验并开始训练
7. ✅ 运行 Step 6 下载结果

**祝训练顺利！** 🚀